In [ ]:
if "worker" in globals() and worker.is_alive():
    raise RuntimeError("A benchmark is already running. Stop it before running setup again.")

import json
import os
import tempfile
import threading
import time
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parents[1]
if not (REPO_ROOT / "benchmarking" / "src").is_dir():
    raise RuntimeError("Start Jupyter from the folder benchmarking/benchmark_runs")

sys.path.insert(0, str(REPO_ROOT))

from benchmarking.src.config import load_config
from benchmarking.src.runner import RunCancelled, run

config = load_config(REPO_ROOT / "benchmarking" / "example.toml")

fd, metrics_name = tempfile.mkstemp(prefix="moku-live-", suffix=".jsonl")
os.close(fd)
metrics_path = Path(metrics_name)

state = {"run_dir": None, "error": None}

In [ ]:
if "worker" in globals() and worker.is_alive():
    raise RuntimeError("A benchmark is already running. Stop it before starting another.")

stop_event = threading.Event()
state = {"run_dir": None, "error": None, "cancelled": False}
metrics_path.write_bytes(b"")

def on_progress(message: str) -> None:
    if message.startswith("Run directory: "):
        state["run_dir"] = Path(message.partition(": ")[2])
    print(message, flush=True)


def run_benchmark() -> None:
    try:
        state["run_dir"] = run(
            config,
            progress=on_progress,
            progress_path=metrics_path,
            stop_event=stop_event,
        )
    except RunCancelled:
        state["cancelled"] = True
    except Exception as exc:
        state["error"] = exc


worker = threading.Thread(target=run_benchmark, daemon=True)

print("Benchmark prepared. Run the plotting cell to start.")

In [ ]:
rows = []
samples_per_engine = {}
searches_per_engine = {}
score_history = []
result_history = []
last_pairs = 0
offset = 0
pending = b""

fig, axes = plt.subplots(4, 2, figsize=(14, 14), constrained_layout=True)
ax_nps, ax_depth = axes[0]
ax_score, ax_results = axes[1]
ax_nodes, ax_elapsed = axes[2]
ax_tt = axes[3, 0]
axes[3, 1].set_visible(False)
plot_handle = display(fig, display_id=True)
plt.close(fig)

try:
    worker.start()
    while worker.is_alive() or offset < metrics_path.stat().st_size:
        with metrics_path.open("rb") as stream:
            stream.seek(offset)
            chunk = stream.read()
            offset = stream.tell()

        pending += chunk
        lines = pending.split(b"\n")
        pending = lines.pop()

        for line in lines:
            if not line:
                continue
            metric = json.loads(line)
            engine = metric["engine"]
            samples_per_engine[engine] = samples_per_engine.get(engine, 0) + 1
            metric["sample"] = samples_per_engine[engine]
            if metric.get("event") == "search_finished":
                searches_per_engine[engine] = searches_per_engine.get(engine, 0) + 1
                metric["search"] = searches_per_engine[engine]
            rows.append(metric)

        run_dir = state["run_dir"]
        summary_path = run_dir / "summary.json" if run_dir else None
        if summary_path and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            pairs = summary.get("pairs", 0)
            if pairs > last_pairs and summary.get("score") is not None:
                score_history.append((pairs, summary["score"] * 100))
                result_history.append(
                    (pairs, summary["wins"], summary["draws"], summary["losses"])
                )
                last_pairs = pairs

        for axis in axes.flat:
            axis.clear()
        axes[3, 1].set_visible(False)

        for engine in samples_per_engine:
            engine_rows = [row for row in rows if row["engine"] == engine]
            nps_rows = [row for row in engine_rows if isinstance(row.get("nps"), (int, float))]
            depth_rows = [
                row for row in engine_rows
                if isinstance(row.get("depth_completed"), (int, float))
            ]
            finished_rows = [
                row for row in engine_rows
                if row.get("event") == "search_finished" and "search" in row
            ]

            if nps_rows:
                ax_nps.plot(
                    [row["sample"] for row in nps_rows],
                    [row["nps"] for row in nps_rows],
                    label=engine,
                )
            if depth_rows:
                ax_depth.plot(
                    [row["sample"] for row in depth_rows],
                    [row["depth_completed"] for row in depth_rows],
                    label=engine,
                )

            node_rows = [
                row for row in finished_rows
                if isinstance(row.get("nodes_total"), (int, float))
            ]
            if node_rows:
                ax_nodes.plot(
                    [row["search"] for row in node_rows],
                    [row["nodes_total"] for row in node_rows],
                    label=engine,
                )

            elapsed_rows = [
                row for row in finished_rows
                if isinstance(row.get("elapsed_ms"), (int, float))
            ]
            if elapsed_rows:
                ax_elapsed.plot(
                    [row["search"] for row in elapsed_rows],
                    [row["elapsed_ms"] for row in elapsed_rows],
                    label=engine,
                )

            tt_rows = [
                row for row in finished_rows
                if isinstance(row.get("tt_hits_total"), (int, float))
                and isinstance(row.get("tt_probes_total"), (int, float))
                and row["tt_probes_total"] > 0
            ]
            if tt_rows:
                ax_tt.plot(
                    [row["search"] for row in tt_rows],
                    [
                        row["tt_hits_total"] / row["tt_probes_total"] * 100
                        for row in tt_rows
                    ],
                    label=engine,
                )

        if score_history:
            ax_score.plot(
                [pair for pair, _ in score_history],
                [score for _, score in score_history],
                marker="o",
            )

        if result_history:
            pairs = [result[0] for result in result_history]
            ax_results.plot(
                pairs,
                [result[1] for result in result_history],
                label="Candidate Wins",
            )
            ax_results.plot(
                pairs,
                [result[2] for result in result_history],
                label="Draws",
            )
            ax_results.plot(
                pairs,
                [result[3] for result in result_history],
                label="Reference Wins",
            )

        ax_nps.set(title="Nodes per Second", xlabel="Metric Sample", ylabel="NPS")
        ax_depth.set(title="Completed Search Depth", xlabel="Metric Sample", ylabel="Depth")
        ax_score.set(
            title="Candidate Score after Complete Opening Pairs",
            xlabel="Pairs",
            ylabel="Score (%)",
            ylim=(0, 100),
        )
        ax_results.set(
            title="Results after Complete Opening Pairs",
            xlabel="Pairs",
            ylabel="Games",
        )
        ax_nodes.set(
            title="Nodes per successful search",
            xlabel="Successful search",
            ylabel="Nodes",
        )
        ax_elapsed.set(
            title="Elapsed time per successful search",
            xlabel="Successful search",
            ylabel="Time (ms)",
        )
        ax_tt.set(
            title="TT Hit Rate per successful search",
            xlabel="Successful search",
            ylabel="Hits / Probes (%)",
            ylim=(0, 100),
        )

        for axis in (
            ax_nps, ax_depth, ax_results, ax_nodes, ax_elapsed, ax_tt
        ):
            if axis.lines:
                axis.legend()

        plot_handle.update(fig)
        time.sleep(1)
finally:
    stop_event.set()
    while worker.is_alive():
        worker.join(timeout=0.1)
    plt.close(fig)
    if state["error"] is not None:
        raise RuntimeError("Benchmark failed; engine cleanup has finished.") from state["error"]

print(f"Benchmark beendet. Metrikdatei: {metrics_path}")
print(f"Ergebnisordner: {state['run_dir']}")